## 1. Imports & Setup

In [ ]:
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, ElasticNet,
    SGDRegressor, SGDClassifier, LogisticRegression,
    LassoCV, LogisticRegressionCV,
)
from sklearn.neural_network import MLPRegressor, MLPClassifier

from sklearn.metrics import (
    r2_score, mean_absolute_error, root_mean_squared_error,
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix, classification_report,
)

from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from scipy.stats import randint, uniform
from xgboost import XGBRegressor, XGBClassifier

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Load Data

In [3]:
df = pd.read_csv("data/cibil_score/cibil_score.csv")
df = df.drop(columns=["Unnamed: 0"])

# normalize column names
df.columns = [col.lower().strip() for col in df.columns]

print(df.shape)
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
df.head()

(51336, 87)
Duplicate rows: 0


,prospectid,total_tl,tot_closed_tl,tot_active_tl,total_tl_opened_l6m,tot_tl_closed_l6m,pct_tl_open_l6m,pct_tl_closed_l6m,pct_active_tl,pct_closed_tl,...,pct_cc_enq_l6m_of_l12m,pct_pl_enq_l6m_of_ever,pct_cc_enq_l6m_of_ever,max_unsec_exposure_inpct,hl_flag,gl_flag,last_prod_enq2,first_prod_enq2,credit_score,approved_flag
0,1,5,4,1,0,0,0.000,0.0,0.200,0.800,...,0.0,0.0,0.0,13.333,1,0,PL,PL,696,P2
1,2,1,0,1,0,0,0.000,0.0,1.000,0.000,...,0.0,0.0,0.0,0.860,0,0,ConsumerLoan,ConsumerLoan,685,P2
2,3,8,0,8,1,0,0.125,0.0,1.000,0.000,...,0.0,0.0,0.0,5741.667,1,0,ConsumerLoan,others,693,P2
3,4,1,0,1,1,0,1.000,0.0,1.000,0.000,...,0.0,0.0,0.0,9.900,0,0,others,others,673,P2
4,5,3,2,1,0,0,0.000,0.0,0.333,0.667,...,0.0,0.0,0.0,-99999.000,0,0,AL,AL,753,P1


In [4]:
df["approved_flag"].value_counts()

approved_flag
P2    32199
P3     7452
P4     5882
P1     5803
Name: count, dtype: int64

## 3. Feature / Target Setup
* `X` -> all columns except `credit_score` and `approved_flag`
* `y_linear` -> `credit_score` (Linear Regression target)
* `y_multiclass` -> `approved_flag` (Logistic Regression, multiclass: P1/P2/P3/P4)
* `y_binary` -> `approved_flag` mapped to **1** for P1/P2 and **0** for P3/P4 (Logistic Regression, binary)

In [5]:
X = df.drop(columns=["approved_flag", "credit_score"])
y_linear = df["credit_score"].astype(float)
y_multiclass = df["approved_flag"].astype(str)

binary_map = {"P1": 1, "P2": 1, "P3": 0, "P4": 0}
y_binary = df["approved_flag"].map(binary_map)

assert y_binary.isna().sum() == 0, "approved_flag has values outside P1-P4"

print(y_multiclass.value_counts())
print(y_binary.value_counts())

approved_flag
P2    32199
P3     7452
P4     5882
P1     5803
Name: count, dtype: int64
approved_flag
1    38002
0    13334
Name: count, dtype: int64


## 4. Train / Test Split
A single split on `X` is created and then reused (via the shared index) for every target so that every model family sees the exact same rows in train/test.

In [6]:
X_train, X_test, idx_train, idx_test = train_test_split(
    X, X.index, test_size=0.2, random_state=RANDOM_STATE
)

y_linear_train, y_linear_test = y_linear.loc[idx_train], y_linear.loc[idx_test]
y_multi_train, y_multi_test = y_multiclass.loc[idx_train], y_multiclass.loc[idx_test]
y_bin_train, y_bin_test = y_binary.loc[idx_train], y_binary.loc[idx_test]

print(X_train.shape, X_test.shape)

(41068, 85) (10268, 85)


## 5. Missing Value Handling
Domain-specific cleanup carried over/expanded from the original notebook. The dataset encodes several kinds of \"missing\" as the sentinel value `-99999`; the correct treatment depends on what the field means.

In [7]:
# 5a. Drop columns with very high missingness
high_missing = [c for c in ["cc_utilization", "pl_utilization"] if c in X_train.columns]
X_train = X_train.drop(columns=high_missing)
X_test = X_test.drop(columns=high_missing)

# 5b. Median-impute a handful of numeric fields where -99999 means "unknown"
median_columns = [
    "age_oldest_tl", "age_newest_tl", "pct_currentbal_all_tl", "time_since_recent_payment",
]
for col in median_columns:
    if col not in X_train.columns:
        continue
    X_train[col] = X_train[col].replace(-99999, np.nan)
    X_test[col] = X_test[col].replace(-99999, np.nan)
    train_median = X_train[col].median()
    X_train[col] = X_train[col].fillna(train_median)
    X_test[col] = X_test[col].fillna(train_median)  # use TRAIN median to avoid leakage

# 5c. Delinquency fields: -99999 means "never delinquent" -> 0
delinquency_columns = [
    "max_delinquency_level", "max_deliq_6mts", "max_deliq_12mts",
    "time_since_recent_deliquency", "time_since_first_deliquency",
]
delinquency_columns = [c for c in delinquency_columns if c in X_train.columns]
X_train[delinquency_columns] = X_train[delinquency_columns].replace(-99999, 0)
X_test[delinquency_columns] = X_test[delinquency_columns].replace(-99999, 0)

# 5d. Enquiry fields: -99999 means "no enquiry" -> 0
enquiry_columns = [
    "tot_enq", "cc_enq", "pl_enq", "cc_enq_l6m", "cc_enq_l12m",
    "pl_enq_l6m", "pl_enq_l12m", "enq_l3m", "enq_l6m", "enq_l12m",
]
enquiry_columns = [c for c in enquiry_columns if c in X_train.columns]
X_train[enquiry_columns] = X_train[enquiry_columns].replace(-99999, 0)
X_test[enquiry_columns] = X_test[enquiry_columns].replace(-99999, 0)

# 5e. time_since_recent_enq: -99999 = "no enquiry ever" -> longer than any observed gap,
# keep the "no enquiry" signal as its own binary flag (fit on TRAIN only, applied to both)
col = "time_since_recent_enq"
if col in X_train.columns:
    train_mask = X_train[col] == -99999
    test_mask = X_test[col] == -99999
    fill_value = X_train.loc[~train_mask, col].max() + 1

    X_train["no_enquiry_flag"] = train_mask.astype(int)
    X_test["no_enquiry_flag"] = test_mask.astype(int)

    X_train[col] = X_train[col].replace(-99999, fill_value)
    X_test[col] = X_test[col].replace(-99999, fill_value)

# 5f. max_unsec_exposure_inpct: -99999 = "no unsecured loan" -> 0%
col = "max_unsec_exposure_inpct"
if col in X_train.columns:
    X_train[col] = X_train[col].replace(-99999, 0)
    X_test[col] = X_test[col].replace(-99999, 0)

# 5g. Log-transform skewed income
if "netmonthlyincome" in X_train.columns:
    X_train["netmonthlyincome_log"] = np.log1p(X_train["netmonthlyincome"].clip(lower=0))
    X_test["netmonthlyincome_log"] = np.log1p(X_test["netmonthlyincome"].clip(lower=0))
    X_train = X_train.drop(columns=["netmonthlyincome"])
    X_test = X_test.drop(columns=["netmonthlyincome"])

print("Remaining NaNs in X_train:", X_train.isna().sum().sum())
print("Remaining NaNs in X_test:", X_test.isna().sum().sum())

Remaining NaNs in X_train: 0
Remaining NaNs in X_test: 0


## 6. Preprocessing Pipeline (scale numeric, one-hot encode categorical)

In [8]:
numeric_columns = X_train.select_dtypes(include=np.number).columns
categorical_columns = X_train.select_dtypes(include="object").columns
print(f"{len(numeric_columns)} numeric columns, {len(categorical_columns)} categorical columns")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_columns),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_columns),
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()
X_train_processed = pd.DataFrame(X_train_processed, columns=feature_names, index=X_train.index)
X_test_processed = pd.DataFrame(X_test_processed, columns=feature_names, index=X_test.index)

print("Processed shapes:", X_train_processed.shape, X_test_processed.shape)

79 numeric columns, 5 categorical columns
Processed shapes: (41068, 102) (10268, 102)


## 7. Hyperparameter Search

Grid Search is being executed for three model families: LinearRegression, RandomForestRegressor, XGBRegressor, Logistic Regression, RandomForest Classifier, XGBoost Classifier

In [ ]:
ALL_RESULTS = []

def evaluate_regression(model, X_te, y_te):
    y_pred = model.predict(X_te)
    return {
        "r2": r2_score(y_te, y_pred),
        "mae": mean_absolute_error(y_te, y_pred),
        "rmse": root_mean_squared_error(y_te, y_pred),
    }

def evaluate_classification(model, X_te, y_te, binary):
    y_pred = model.predict(X_te)
    metrics = {
        "accuracy": accuracy_score(y_te, y_pred),
        "f1_macro": f1_score(y_te, y_pred, average="macro", zero_division=0),
        "precision_macro": precision_score(y_te, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_te, y_pred, average="macro", zero_division=0),
        # Weighted averages
        "precision_weighted": precision_score(
            y_te, y_pred, average="weighted", zero_division=0
        ),
        "recall_weighted": recall_score(
            y_te, y_pred, average="weighted", zero_division=0
        ),
        "f1_weighted": f1_score(
            y_te, y_pred, average="weighted", zero_division=0
        ),
        # Confusion Matrix
        "confusion_matrix": confusion_matrix(y_te, y_pred)        
    }
    if binary and hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_te)[:, 1]
        metrics["roc_auc"] = roc_auc_score(y_te, y_prob)
    return metrics

In [13]:
HPARAM_RESULTS = []

# Lighter, 3-fold CV specifically for hyperparameter search (Section 14's 5-fold kfold/skfold
# are reused everywhere else; tuning many models x many candidates is far more expensive, so a
# smaller cv here keeps runtime reasonable). Bump back to 5 if you have the compute budget.
search_kfold = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
search_skfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
RANDOM_SEARCH_N_ITER = 10  # number of sampled combinations for RandomizedSearchCV

# XGBoost needs 0..n_classes-1 integer labels for multiclass
label_encoder = LabelEncoder()
y_multi_train_enc = label_encoder.fit_transform(y_multi_train)
y_multi_test_enc = label_encoder.transform(y_multi_test)
print("Multiclass label mapping:", dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

Multiclass label mapping: {'P1': 0, 'P2': 1, 'P3': 2, 'P4': 3}


In [15]:
def linear_model_specs():
    """model_name -> (estimator, grid_search_param_grid, random_search_param_distributions)"""
    return {
        "linear_regression": (
            LinearRegression(),
            {"fit_intercept": [True, False], "positive": [True, False]},
            {"fit_intercept": [True, False], "positive": [True, False]},
        ),
        "random_forest": (
            RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=1),
            {
                "n_estimators": [100, 150],
                "max_depth": [5, 10],
            },
            {
                "n_estimators": randint(50, 250),
                "max_depth": [3, 5, 8, None],
                "min_samples_leaf": randint(1, 6),
            },
        ),
        "xgboost": (
            XGBRegressor(random_state=RANDOM_STATE, verbosity=0, n_jobs=1),
            {
                "n_estimators": [100, 150],
                "max_depth": [3, 5],
            },
            {
                "n_estimators": randint(50, 250),
                "max_depth": randint(2, 8),
                "learning_rate": uniform(0.03, 0.27),
                "subsample": uniform(0.6, 0.4),
            },
        ),
    }


def classification_model_specs():
    """model_name -> (estimator, grid_search_param_grid, random_search_param_distributions)"""
    return {
        "logistic_regression": (
            LogisticRegression(solver="saga", max_iter=2000, random_state=RANDOM_STATE),
            {"C": [0.1, 1.0, 10.0], "l1_ratio": [0.0, 0.5, 1.0]},
            {"C": uniform(0.01, 15), "l1_ratio": uniform(0, 1)},
        ),
        "random_forest": (
            RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1),
            {
                "n_estimators": [100, 150],
                "max_depth": [5, 10],
            },
            {
                "n_estimators": randint(50, 250),
                "max_depth": [3, 5, 8, None],
                "min_samples_leaf": randint(1, 6),
            },
        ),
        "xgboost": (
            XGBClassifier(random_state=RANDOM_STATE, verbosity=0, eval_metric="logloss", n_jobs=1),
            {
                "n_estimators": [100, 150],
                "max_depth": [3, 5],
            },
            {
                "n_estimators": randint(50, 250),
                "max_depth": randint(2, 8),
                "learning_rate": uniform(0.03, 0.27),
                "subsample": uniform(0.6, 0.4),
            },
        ),
    }

### GridSearchCV for Linear Regression

### GridSearchSF for Logistic Regression binary classification

How to choose CV vs SF?

If case of imbalance among target variables, we use Startified fold else, use kfold

As we have imbalance for P1 class so will use SF


### RandomSearchSF for Logistic Regression binary classification

### RandomSearchSF for Logistic Regression multi classification